In [ ]:
SYSTEM_PROMPT = """
You are a powerful personal AI assistant to help user with managing his daily meetings
You have access to his meeting schedules using tools.
You need to entertain users requests and queries with updated knowledge and information.
You can perform actions, interact to external environment using tool/function calling.
"""

In [ ]:
DEVELOPER_PROMPT = """
Use the required tool to complete the request. If tools cannot fulfill user requests, simply apologies and Do not come up with false information or unhelpful results.
You are required to return the response in specific JSON format.
{
content: "str, content messages",
len: "int, total events for that day"
}
"""

In [ ]:
user_query_1 = """
I want to know about my calendar schedule on friday
"""

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

In [ ]:
BASE_URL = "https://openrouter.ai/api/v1"
MODEL = "dots-studio/dots-3-note-preview:free"

In [ ]:
from openai import AsyncOpenAI

client = AsyncOpenAI(api_key=OPENROUTER_API_KEY, base_url=BASE_URL)

In [ ]:
def AI(messages: list, tools=[]):
    chat_completion = client.chat.completions.create(
        messages=messages, tools=tools, model=MODEL
    )
    return chat_completion

In [ ]:
def user_calendar(day: str):
    calendars = {
        "monday": {
            "meetings": [{"title": "Team Standup", "time": "09:00"}],
            "schedules": [{"title": "Focus Time", "start": "10:00", "end": "12:00"}],
        },
        "tuesday": {
            "meetings": [{"title": "Client Meeting", "time": "11:00"}],
            "schedules": [{"title": "Project Work", "start": "13:00", "end": "16:00"}],
        },
        "wednesday": {
            "meetings": [{"title": "Project Review", "time": "14:00"}],
            "schedules": [{"title": "Focus Time", "start": "09:00", "end": "12:00"}],
        },
        "thursday": {
            "meetings": [{"title": "Planning", "time": "10:00"}],
            "schedules": [{"title": "Development", "start": "13:00", "end": "17:00"}],
        },
        "friday": {
            "meetings": [{"title": "Weekly Review", "time": "15:00"}],
            "schedules": [{"title": "Admin Work", "start": "09:00", "end": "11:00"}],
        },
        "saturday": {"meetings": [], "schedules": []},
        "sunday": {"meetings": [], "schedules": []},
    }

    return {
        "day": day,
        **calendars.get(day.lower(), {"meetings": [], "schedules": []}),
    }

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "user_calendar",
            "description": "Get user calendar and meeting info",
            "parameters": {
                "type": "object",
                "properties": {
                    "day": {
                        "type": "string",
                        "description": "Week day parameter to get calendar data on specified day",
                    }
                },
                "required": ["day"],
            },
        },
    }
]

In [ ]:
tools_func_map = {"user_calendar": user_calendar}

In [ ]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "developer", "content": DEVELOPER_PROMPT},
    {"role": "user", "content": user_query_1},
]

In [ ]:
ai_response = await AI(messages, tools)

In [ ]:
parsed = ai_response.model_dump()

In [ ]:
assistant_reply = ai_response.choices[0].message.model_dump()

In [ ]:
messages.append(assistant_reply)

In [ ]:
if parsed_message := parsed["choices"][0].get("message"):
    if tool_call := parsed_message.get("tool_calls"):
        func_to_call = tool_call[0].get("function")
        tool_call_name = func_to_call.get("name")
        tool_call_args = func_to_call.get("arguments")
        tool_call_id = tool_call[0].get("id")

In [ ]:
def handle_tool_calls(func_name, args):
    func_call = tools_func_map.get(func_name)

    if not func_call:
        raise ValueError("Unhandled function call")
    try:
        import json

        args = json.loads(args)
        func_response = func_call(**args)
        return {
            "metadata": f"tool call [{func_name}] is successful and output is available in response key",
            "response": func_response,
            "success": True,
            "func_name": func_name,
        }
    except Exception as e:
        print("failed to parse args")
        return {
            "metadata": f"tool call [{func_name}] is unsuccessful. Unable to provide response",
            "success": False,
            "func_name": func_name,
        }

In [ ]:
tool_output = handle_tool_calls(tool_call_name, tool_call_args)

In [ ]:
formatted_tool_output = {
    "role": "tool",
    "tool_name": tool_output.get("func_name", tool_call_name),
    "tool_id": tool_call_id,
    "content": f"Function metadata: {tool_output.get("metadata")}. Function completion status: {tool_output.get("success")}. Function Response: {tool_output.get("response")}",
}

In [ ]:
messages.append(formatted_tool_output)

In [ ]:
messages

In [ ]:
ai_response = await AI(messages)


In [ ]:
messages.append(ai_response.choices[0].message.model_dump())

In [ ]:
messages

In [ ]:
async def agentic_flow(user_query: str):
    MESSAGES = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "developer", "content": DEVELOPER_PROMPT},
        {"role": "user", "content": user_query},
    ]
    try:
        ai_response = await AI(messages, tools)
        
    except Exception as excep:
        print("Unhandled exception in agentic flow")

In [ ]:
async def reflection(nb:int, day:str):
    # ai_response = await AI(messages, tools)
    # check for total meetings on that day and compare
    pass